In [1]:
import json
import os
try:
    from backend.utils import load_json
except ModuleNotFoundError:
    print("Updating working directory....")
    os.chdir("../")
    # print(os.getcwd())
    from backend.utils import load_json

from pathlib import Path

Updating working directory....


In [2]:
output_dir = Path("out/responses/base_tempsweep")
model_dirs = [p for p in output_dir.iterdir() if p.is_dir()]
print(model_dirs)

[PosixPath('out/responses/base_tempsweep/ministral-3_14b'), PosixPath('out/responses/base_tempsweep/gemma4_e2b'), PosixPath('out/responses/base_tempsweep/ministral-3_8b'), PosixPath('out/responses/base_tempsweep/gemma4_26b'), PosixPath('out/responses/base_tempsweep/qwen3.5_4b_nothink'), PosixPath('out/responses/base_tempsweep/ministral-3_3b'), PosixPath('out/responses/base_tempsweep/qwen3.5_4b'), PosixPath('out/responses/base_tempsweep/gemma4_e4b'), PosixPath('out/responses/base_tempsweep/qwen3.6_35b'), PosixPath('out/responses/base_tempsweep/qwen3.5_27b'), PosixPath('out/responses/base_tempsweep/qwen3.6_35b_nothink'), PosixPath('out/responses/base_tempsweep/qwen3.5_9b')]


In [3]:
def match_substr(substr, exclude=None):
    return [p for p in model_dirs if (substr in str(p) and (exclude is None or exclude not in str(p)))]

gemma4 = match_substr("gemma4")
qwen35 = match_substr("qwen3.5")
qwen36 = match_substr("qwen3.6", exclude="nothink")
ministral3 = match_substr("ministral-3")

by_family = [gemma4, qwen35, qwen36, ministral3]
print(by_family)

[[PosixPath('out/responses/base_tempsweep/gemma4_e2b'), PosixPath('out/responses/base_tempsweep/gemma4_26b'), PosixPath('out/responses/base_tempsweep/gemma4_e4b')], [PosixPath('out/responses/base_tempsweep/qwen3.5_4b_nothink'), PosixPath('out/responses/base_tempsweep/qwen3.5_4b'), PosixPath('out/responses/base_tempsweep/qwen3.5_27b'), PosixPath('out/responses/base_tempsweep/qwen3.5_9b')], [PosixPath('out/responses/base_tempsweep/qwen3.6_35b')], [PosixPath('out/responses/base_tempsweep/ministral-3_14b'), PosixPath('out/responses/base_tempsweep/ministral-3_8b'), PosixPath('out/responses/base_tempsweep/ministral-3_3b')]]


In [5]:
for num, fam in enumerate(by_family):
    rows = [list(), list(), list(), list(), list()]
    for md in (fam):
        model_dir = (md) / "narendra_nli_nonbinary_concise"
        temp_dirs = sorted([p for p in model_dir.iterdir() if p.is_dir()])

        for j, temp_dir in enumerate(temp_dirs):
            temp = (str(temp_dir)[-3:])

            if len(rows[j]) == 0:
                rows[j].append(temp)
            else:
                if rows[j][0] != temp:
                    print(rows[j][0], temp)
                    raise ValueError("What the fuck?")
                
            metrics_file = temp_dir / "eval" / "aggregated_metrics.json"
            metrics = load_json(metrics_file)
            accuracy = metrics["macro_label_micro_doc"]["class"]["f1_entailment"]
            acc_avg = accuracy["average"]
            acc_std = accuracy["std"]
            
            rows[j].append(str(acc_avg))

            # print(temp, acc_avg)    
    
        header = "y " + " ".join([f"x{i}" for i in range(len(fam))]) + "\n"
        rows_str = [" ".join(row) for row in rows]
        out_str = header + "\n".join(rows_str)
        out_str += "\n% " + "\n% ".join([str(m) for m in fam])
    print(out_str)

    os.makedirs(".f1e",exist_ok=True)
    with open(f".f1e/{num}.txt", "w") as f:
        f.write(out_str)
    # _ = [print(r) for r in rows]

y x0 x1 x2
0.1 0.6764955598664555 0.807082059993252 0.7511696562640782
0.2 0.6752328380927931 0.8051898890618822 0.7507877693903234
0.3 0.6811132897010467 0.8078938805392676 0.7530336140808225
0.4 0.6759175428857382 0.8024433445725471 0.7497568720048058
0.5 0.6781332532199503 0.8062346787649135 0.7459022197962789
% out/responses/base_tempsweep/gemma4_e2b
% out/responses/base_tempsweep/gemma4_26b
% out/responses/base_tempsweep/gemma4_e4b
y x0 x1 x2 x3
0.1 0.7347999328875011 0.7507580674345798 0.785967766106949 0.7616274374331025
0.2 0.7361246018496672 0.7368455337112397 0.7816013477849744 0.7522972053442438
0.3 0.7424455967031868 0.7371999886234822 0.7826308678785563 0.7490720866707012
0.4 0.7407069532183933 0.7390873860533204 0.7852580334926422 0.7514193505312506
0.5 0.7306752396172586 0.7331410870948573 0.7858635652120741 0.7438470509781503
% out/responses/base_tempsweep/qwen3.5_4b_nothink
% out/responses/base_tempsweep/qwen3.5_4b
% out/responses/base_tempsweep/qwen3.5_27b
% out/respo